# Code Lab: Time Series Models & Pre-processing

In this lab, we will explore the three tiers of Time Series modeling:
1. **ARIMA (Statistical):** Good for simple, stable datasets.
2. **Facebook Prophet (Business AI):** Excellent for daily data with holidays, trends, and missing gaps.
3. **LSTM (Deep Learning):** Powerful for complex, high-dimensional patterns with long-term dependencies.

In [ ]:
# Step 1: Install require dependencies
# Note: If 'prophet' fails to install on your local Windows machine due to C++ compile errors,
# simply run this notebook in Google Colab where Prophet is pre-installed!
!pip install prophet pandas numpy matplotlib statsmodels

### Step 2: Generating Sample Time Series Data
We will synthesize a fake dataset representing Ice Cream sales over 3 years, embedding an obvious summer "Seasonality" spike and a steady 10% upward "Trend."

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Generate 3 years of daily dates
dates = pd.date_range(start='2020-01-01', end='2022-12-31', freq='D')

# Create synthetic data: linear trend + sine wave (summer spikes) + random noise
trend = np.linspace(10, 50, len(dates))
seasonality = 20 * np.sin(np.pi * dates.dayofyear / 182.5) # Peaks in summer
noise = np.random.normal(0, 5, len(dates))

sales = trend + seasonality + noise

df = pd.DataFrame({'Date': dates, 'Sales': sales})
df.set_index('Date', inplace=True)

plt.figure(figsize=(12, 4))
plt.plot(df.index, df['Sales'], color='blue', alpha=0.6)
plt.title('Ice Cream Sales (3 Years)')
plt.ylabel('Units Sold')
plt.show()

### Step 3: Facebook's Prophet Model
Prophet requires our DataFrame to have specific column names: `ds` for dates, and `y` for the target variable.

In [ ]:
from prophet import Prophet

# Prepare data for Prophet
prophet_df = df.reset_index()
prophet_df = prophet_df.rename(columns={'Date': 'ds', 'Sales': 'y'})

# Initialize and train the model
# We tell it we want yearly seasonality to be closely monitored
m = Prophet(yearly_seasonality=True, daily_seasonality=False)
m.fit(prophet_df)

In [ ]:
# Predict 1 year into the future (365 days)
future = m.make_future_dataframe(periods=365)
forecast = m.predict(future)

# Plot the forecast
fig1 = m.plot(forecast)
plt.title("Prophet Forecast (Black dots = Actual, Blue line = Prediction)")
plt.show()

#### The Magic of Prophet: Component Tracking
Prophet has a built-in function to mathematically separate the underlying Trend from the Seasonality. This makes it highly readable for business executives.

In [ ]:
fig2 = m.plot_components(forecast)
plt.show()

### Step 4: LSTM (Memory Gates in Deep Learning)
For incredibly complex patterns (like Stock Market algorithms checking 100 different features), we use LSTMs. We must create a sliding "Lookback Window" block.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Lookback Window configuration
time_steps = 10 # Predict Day 11 by looking at Days 1-10

def create_sequences(data, time_steps):
    X_seq, y_seq = [], []
    for i in range(len(data) - time_steps):
        X_seq.append(data[i:(i + time_steps)])
        y_seq.append(data[i + time_steps])
    return np.array(X_seq), np.array(y_seq)

data_values = df['Sales'].values
X_lstm, y_lstm = create_sequences(data_values, time_steps)

# Keras LSTM requires 3D specific shape (Samples, Timesteps, Features)
X_lstm = X_lstm.reshape((X_lstm.shape[0], X_lstm.shape[1], 1))

print(f"X shape (Sequences): {X_lstm.shape}")
print(f"y shape (Targets): {y_lstm.shape}")

In [ ]:
# Build the LSTM "Memory Cell" Brain
model = Sequential([
    LSTM(50, activation='relu', input_shape=(time_steps, 1)), # 50 memory cells
    Dense(1) # Final predicted value
])

model.compile(optimizer='adam', loss='mse')

# Train for 5 epochs to demonstrate learning capacity
model.fit(X_lstm, y_lstm, epochs=5, verbose=1)
print("LSTM Model trained successfully!")